## T Test; CI; SE

In [ ]:
# Step 0: imports
import numpy as np
import pandas as pd
from statsmodels.stats.power import NormalIndPower
from scipy.stats import ttest_ind

# -------------------------------
# Step 1: Create raw data
# -------------------------------

# Assumptions
n_users_per_group = 8000          # sample size per group
baseline_apply_rate = 0.08        # 5% apply rate in control
mde = 0.02                        # target minimum detectable effect (absolute lift)

# Create user IDs
control_users = pd.DataFrame({'user_id': range(n_users_per_group),
                              'variant': 'control'})
treatment_users = pd.DataFrame({'user_id': range(n_users_per_group),
                                'variant': 'treatment'})

control_users['applied'] = np.random.binomial(1, baseline_apply_rate, n_users_per_group)
treatment_users['applied'] = np.random.binomial(1, baseline_apply_rate+mde, n_users_per_group)
users = pd.concat([control_users, treatment_users]).reset_index(drop=True)

# Simulate application outcome
# Control: baseline 8%, Treatment: baseline + MDE


# Quick check
users.head()



,user_id,variant,applied
0,0,control,0
1,1,control,0
2,2,control,0
3,3,control,0
4,4,control,0


In [39]:
x = users.groupby('variant')['applied'].agg(['sum', 'count','mean'])
x

,sum,count,mean
variant,,,
control,611,8000,0.076375
treatment,809,8000,0.101125


In [44]:
x.loc['control','sum']

np.int64(611)

In [37]:
users.groupby('variant')['applied'].count()

variant
control      8000
treatment    8000
Name: applied, dtype: int64

In [31]:
# -------------------------------
# Step 2: Power analysis
# -------------------------------

analysis = NormalIndPower()

# calculate power given n, alpha, effect size
effect_size = mde / np.sqrt(baseline_apply_rate*(1-baseline_apply_rate))  # Cohen's h approx
alpha = 0.05

power = analysis.solve_power(effect_size=effect_size, nobs1=n_users_per_group, alpha=alpha, ratio=1.0, alternative='two-sided')
print(f"Power with {n_users_per_group} users per group: {power:.2f}")

# If we want required sample size for 80% power
n_required = analysis.solve_power(effect_size=effect_size, power=0.8, alpha=alpha, ratio=1.0, alternative='two-sided')
print(f"Required sample size per group for 80% power: {np.ceil(n_required)}")




Power with 8000 users per group: 1.00
Required sample size per group for 80% power: 2889.0


In [32]:
# -------------------------------
# Step 3: Run test (simulate experiment)
# -------------------------------

# Calculate apply rate per group
summary = users.groupby('variant')['applied'].agg(['mean','count'])
summary['se'] = np.sqrt(summary['mean']*(1-summary['mean'])/summary['count'])
print(summary)

# (p * (1-p)/ n)**1/2



               mean  count        se
variant                             
control    0.076375   8000  0.002969
treatment  0.101125   8000  0.003371


In [14]:
((0.048875*(1-0.048875))/8000)**(1/2)

0.002410555806629459

In [33]:
# -------------------------------
# Step 4: Statistical test
# -------------------------------

control_apply = users.loc[users['variant']=='control', 'applied']
treatment_apply = users.loc[users['variant']=='treatment', 'applied']

t_stat, p_value = ttest_ind(treatment_apply, control_apply, equal_var=False)
print(f"T-statistic: {t_stat:.3f}, p-value: {p_value:.4f}")



T-statistic: 5.509, p-value: 0.0000


In [6]:
users.head()

,user_id,variant,applied
0,0,control,0
1,1,control,1
2,2,control,0
3,3,control,0
4,4,control,0


In [8]:
# -------------------------------
# Step 5: Analyze results
# -------------------------------

apply_rate_control = control_apply.mean()
apply_rate_treatment = treatment_apply.mean()
lift = apply_rate_treatment - apply_rate_control
ci_low = lift - 1.96 * np.sqrt(summary.loc['control','se']**2 + summary.loc['treatment','se']**2)
ci_high = lift + 1.96 * np.sqrt(summary.loc['control','se']**2 + summary.loc['treatment','se']**2)

print(f"Control apply rate: {apply_rate_control:.3%}")
print(f"Treatment apply rate: {apply_rate_treatment:.3%}")
print(f"Absolute lift: {lift:.3%}")
print(f"95% CI for lift: [{ci_low:.3%}, {ci_high:.3%}]")
if p_value < 0.05:
    print("Result is statistically significant!")
else:
    print("Result is not statistically significant.")

Control apply rate: 4.888%
Treatment apply rate: 0.000%
Absolute lift: -4.888%
95% CI for lift: [-5.360%, -4.415%]
Result is statistically significant!


In [4]:
# Imports
import numpy as np
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# -------------------------------
# Step 1: Define parameters
# -------------------------------
baseline_rate = 0.05  # 5% baseline apply rate
mde = 0.01            # 1% absolute lift
alpha = 0.05          # significance level
power = 0.8           # desired statistical power

# Calculate effect size (Cohen's h)
effect_size = proportion_effectsize(baseline_rate, baseline_rate + mde)

# -------------------------------
# Step 2: Calculate required sample size per group
# -------------------------------
analysis = NormalIndPower()
n_required = analysis.solve_power(effect_size=effect_size, 
                                  power=power, 
                                  alpha=alpha, 
                                  ratio=1.0, 
                                  alternative='two-sided')

print(f"Required sample size per group for detecting {mde*100:.1f}% lift: {np.ceil(n_required)} users")


Required sample size per group for detecting 1.0% lift: 8143.0 users


## DID

In [51]:
import pandas as pd

df = pd.DataFrame({
    'group': ['treatment', 'treatment', 'control', 'control'],
    'period': ['before', 'after', 'before', 'after'],
    'y': [10, 15, 8, 9]
})


In [52]:
df

,group,period,y
0,treatment,before,10
1,treatment,after,15
2,control,before,8
3,control,after,9


In [ ]:
df['treat'] = (df['group'] == 'treatment').astype(int) 
df['post'] = (df['period'] == 'after').astype(int)
df['treat_post'] = df['treat'] * df['post'] #treat * post 的 相乘 是核心，因为它表示 “同时满足两个条件” 的那一组。


In [54]:
df

,group,period,y,treat,post,treat_post
0,treatment,before,10,1,0,0
1,treatment,after,15,1,1,1
2,control,before,8,0,0,0
3,control,after,9,0,1,0


In [62]:
df_new = pd.concat([df]*5,ignore_index = True)
df_new

,group,period,y,treat,post,treat_post
0,treatment,before,10,1,0,0
1,treatment,after,15,1,1,1
2,control,before,8,0,0,0
3,control,after,9,0,1,0
4,treatment,before,10,1,0,0
5,treatment,after,15,1,1,1
6,control,before,8,0,0,0
7,control,after,9,0,1,0
8,treatment,before,10,1,0,0
9,treatment,after,15,1,1,1


In [64]:
import statsmodels.formula.api as smf

model = smf.ols('y ~ treat + post + treat_post', data=df_new).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 5.446e+30
Date:                Sun, 16 Nov 2025   Prob (F-statistic):          2.82e-240
Time:                        15:50:42   Log-Likelihood:                 642.80
No. Observations:                  20   AIC:                            -1278.
Df Residuals:                      16   BIC:                            -1274.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      8.0000   1.33e-15      6e+15      0.0

## SRM

In [67]:
import pandas as pd
from scipy.stats import chi2_contingency

# example dataframe
df = pd.DataFrame({
    'user_id': range(1, 11),
    'group': ['A', 'A', 'A', 'B', 'B', 'B', 'A', 'B', 'A', 'B'],
    'country': ['US','US','CA','US','CA','CA','US','CA','US','US']
})

df_new = pd.concat([df]*100,ignore_index = True)


# create a contingency table
contingency_table = pd.crosstab(df_new['group'], df_new['country'])


# run chi-square test
chi2, p, dof, expected = chi2_contingency(contingency_table)

print(f"\nchi2 statistic: {chi2:.4f}")
print(f"p-value: {p:.4f}")
print(f"degrees of freedom: {dof}")
print("expected frequencies:\n", expected)

# interpretation
if p > 0.05:
    print("\n✅ No significant difference — randomization looks good")
else:
    print("\n❌ Significant difference — check SRM")


chi2 statistic: 165.0042
p-value: 0.0000
degrees of freedom: 1
expected frequencies:
 [[200. 300.]
 [200. 300.]]

❌ Significant difference — check SRM


In [ ]:
contingency_table.head()

country,CA,US
group,,
A,100,400
B,300,200


In [71]:
import pandas as pd
import statsmodels.api as sm

# Example dataset
df = pd.DataFrame({
    'treated': [1,0,1,0,1,0],
    'applies_post': [5,3,6,2,7,3],
    'applies_pre': [4,2,5,2,6,3]
})

# Step 1: regress post-metric on pre-metric
X = sm.add_constant(df['applies_pre'])
model = sm.OLS(df['applies_post'], X).fit()
theta = model.params['applies_pre']

# Step 2: compute CUPED-adjusted outcome
df['applies_cuped'] = df['applies_post'] - theta * df['applies_pre']

# Step 3: run A/B test on adjusted metric
mean_treated = df.loc[df['treated']==1, 'applies_cuped'].mean()
mean_control = df.loc[df['treated']==0, 'applies_cuped'].mean()
lift = mean_treated - mean_control
print("CUPED-adjusted lift:", lift)


CUPED-adjusted lift: 0.19999999999999912


In [72]:
df

,treated,applies_post,applies_pre,applies_cuped
0,1,5,4,0.300
1,0,3,2,0.650
2,1,6,5,0.125
3,0,2,2,-0.350
4,1,7,6,-0.050
5,0,3,3,-0.525
